THIS PROGRAM EXPLORES THE USE OF A BASIC PINN DESIGN TO COMPARE AGAINST AN ANALYTIC SOLUTION TO NEWTON'S COOLING LAW DIFF EQN:dT/dt = k*(Tamb-T): 

In [1]:
#Importing the necessary libraries:
import tensorflow as tf
import numpy as np
import math as mt
import matplotlib.pyplot as plt

In [2]:
#Defining the initial boundary condition values and constants:
T_amb=300;
T_0=523;
k=0.45;
T_amb_norm=(T_amb-T_amb)/223;
T_0_norm=(T_0-T_amb)/223;

In [3]:
# Creating the PINN model architecture for later use:
def create_model():
    model= tf.keras.Sequential();
    model.add(tf.keras.layers.Dense(64,activation="tanh",name="layer_1"));
    model.add(tf.keras.layers.Dense(32,activation="tanh",name="layer_2"));
    model.add(tf.keras.layers.Dense(1,activation=None,name="output_layer")); # Becasue only one y(x) value is being evaluated for a given x.

    return model

In [4]:
#Calling the model to process on the input data in a layer by layer outputting manner and produce a predicted value:
def call_model(model,t):
    t=model.get_layer("layer_1")(t);
    t=model.get_layer("layer_2")(t);
    T_pred=model.get_layer("output_layer")(t);

    return T_pred

In [5]:
# Creating the pde function to calculate the derivative terms and return the residual of the ODE: y''+(pi**2)*sin(pi*x) = 0, 
#considering the 2nd derivative of y_pred:
def pde(model,t):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(t); # x here is an outside tensor value which needs to be explicitely watched.
        T_pred=call_model(model,t); # Further used in computational flow in y' evaluation.
        T_t=tape.gradient(T_pred,t); # Further used in the computational flow in y'' value
    
    T_t=tf.cast(T_t,tf.float32); #typecasting tensor object to float32 type for prevention of mathematical inconsistencies in furthur bodies. 
    
    del tape; #For preventing data leakage during long training runs, since for persistent=True the tape memeory remembers the old values for as long as tape exists
    
    return T_t-tf.cast((k)*(T_amb_norm-T_pred),tf.float32)

In [6]:
#Creating a loss function for calculating the net loss result considering the pde_loss and the bc_loss: 
def loss(model,t,t_bc,T_bc):
    res_pde=pde(model,t); #residual for partial diff equation/ordinary diff eqn
    T_pred_bc=call_model(model,t_bc); #output for the model computation used in the loss residual def

    # Force conversion to float data types for all values under the loss function:
    res_pde=tf.cast(res_pde,tf.float32);
    
    loss_pde=tf.reduce_mean(tf.square(res_pde)); #the right MSE usage in SCIML--> tf.reduce_mean(tf.square())
    loss_bc=tf.reduce_mean(tf.square(T_pred_bc-T_bc));

    return loss_pde+loss_bc

In [7]:
#Creating an optimzer structure with a learning schedule for further development of various other code bodies -- in 'train_model_step' and others:
learning_schedule= tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.0005, # l_0
    decay_steps=1000, # n_steps for entire staircase width--> after n_steps, the learning rate decays by the below rate factor once
    decay_rate=0.96, #alpha
    staircase=True # l0*alpha*floor(step/n_steps)=new_learning_rate --> discrete values and not continuous.
)
optimizer=tf.keras.optimizers.Adam(
    learning_rate=learning_schedule,
    beta_1=0.9, # For the mean of the gradients : m  = beta_1·m  + (1−beta_1)·gradient_current
    beta_2=0.99, # For the variance of the gradients: v=  beta_2·v  + (1−beta_2)·gradient_current²
    epsilon=1e-7, # For taking care of divison by 0 in the updates of the trainable variables: theta  = theta − learn_rate * m̂ / (√v̂ + ε) --> parameter update
    clipnorm=1.0 # For rescaling the entire gradient vector of gradients before apply.gradient() function for hyperparameter tuning 
)

In [8]:
#Creating a function for training the model on the data per epoch iteration:
def train_model_step(model,t_train,t_bc,T_bc):
    with tf.GradientTape() as tape:
        T_pred=call_model(model,t_train); # Coming under the recording envelope of GradientTape class, y_pred and loss are being watched.
        loss_val=loss(model,t_train,t_bc,T_bc);

    grads=tape.gradient(loss_val,model.trainable_variables); # Backpropagation to find the gradients of the loss function with the trainable values (theta) of the model
    
    optimizer.apply_gradients(zip(grads,model.trainable_variables)); #optimzing/tuning the trainable variables (hyperparameters) using the gradients so obtained from the loss function def.

    loss_fin=loss_val;
    return loss_fin

MAIN BODY FOR THE CORE PINN LOGIC FLOW:

In [ ]:
#Forming the data set for the input main values x, and boundary values, x_bc, y_bc:
t_initial=np.linspace(0,10,300).reshape(-1,1);
t_train=tf.convert_to_tensor(t_initial);

t_bc=np.array([[0]],dtype=np.float32);
T_bc=np.array([[T_0_norm]],dtype=np.float32);
t_bc=tf.convert_to_tensor(t_bc); #To be used as tensor objects in further train_model_step() functions.
T_bc=tf.convert_to_tensor(T_bc);

#Making the model and storing it in an object:
model=create_model();

epoch=5000; #Define epoch run iterations

for epoch_iter in range(epoch):
    loss_val=train_model_step(model,t_train,t_bc,T_bc);
    #Outputting only limited iterations since epoch can go to thousands of iterations.
    if epoch_iter == 0 or (epoch_iter) % 500 == 0 or epoch_iter == epoch-1: 
        print(f"For iteration: {epoch_iter} the loss value: {loss_val.numpy()} \n");

For iteration: 0 the loss value: 1.050313115119934 

For iteration: 500 the loss value: 0.00018818657554220408 



In [ ]:
#Testing values on the PINN model:
t_test=np.linspace(0,10,1000).reshape(-1,1);
t_test=tf.convert_to_tensor(t_test);
T_pred=call_model(model,t_test).numpy(); #prediction from test values fed into model coming in normalised form.

T_pred_denorm=T_pred*(T_0-T_amb)+T_amb; #Denormalising for physical values print.

T_true=T_amb+(T_0-T_amb)*np.exp(-k*t_test); #Analytic solution for comparison

In [ ]:
#For plotting the outputs of the PINN compariosn with actual analytic solution:
plt.figure(figsize=(10,6));
plt.plot(t_test,(T_pred_denorm-273),'r',label='PINN Values');
plt.plot(t_test,(T_true-273),'b--',label='Analytic Values');
plt.legend(loc="upper right");
plt.xlabel('Time_values: t (s)');
plt.ylabel('TEMP_values: T (deg C)')
plt.show();